# P00 필터 + 차급(LargeCategoryId)별 분리 저장

- 매칭: 파일명(확장자 제외) 동일 → 이미지 1개 ↔ json 1개
- 유지 기준: `REQUIRE_P00 = True`면 json에 `"classId": "P00.차량전체"` 있는 쌍만
- 분리 기준: json `rawDataInfo.LargeCategoryId` (경차 / 소형차 / 중형차 / 대형차)
- 저장: `DEST_ROOT/<차급>/원천데이터/...`, `DEST_ROOT/<차급>/라벨링데이터/...` (원본 폴더 구조 유지)

> ⚠️ 실제로 옮기려면 1번 셀에서 **`DRY_RUN = False`** 로 바꾼 뒤 6번 셀을 실행하세요. `True`면 미리보기만 합니다.

## 0. 환경 / 라이브러리

In [1]:
import os, sys, json, shutil
from collections import Counter, defaultdict
from pathlib import Path

from tqdm import tqdm

print("Python:", sys.version.split()[0])

Python: 3.11.9


## 1. 경로 & 옵션 설정 (여기만 수정)

In [2]:
# ==== 경로 3개 수정 ====
IMAGE_ROOT = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터")
LABEL_ROOT = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\라벨링데이터")
DEST_ROOT  = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터")
# =======================

TARGET_CLASS  = "P00.차량전체"
REQUIRE_P00   = True     # True: P00 있는 쌍만 / False: 차급만 보고 전부 분리
VALID_CLASSES = ["경차", "소형차", "중형차", "대형차"]   # 이 외 값/누락은 '미분류'로

# --- 동작 옵션 ---
DRY_RUN = False      # True: 미리보기 / False: 실제 실행  ← 실행할 땐 False!
ACTION  = "copy"    # "copy": 원본 보존 복사 / "move": 원본에서 이동

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

## 2. 파일 수집 + 매칭

In [3]:
def find_all_files(root: Path, exts: set):
    if not root.exists():
        raise FileNotFoundError(f"경로가 존재하지 않습니다: {root}")
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]

image_files = find_all_files(IMAGE_ROOT, IMAGE_EXTS)
label_files = find_all_files(LABEL_ROOT, {".json"})
label_lookup = {p.stem: p for p in label_files}

paired = [(img, label_lookup[img.stem]) for img in image_files if img.stem in label_lookup]

print(f"이미지 {len(image_files):,} · 라벨 {len(label_files):,} · 매칭된 쌍 {len(paired):,}")

이미지 257,740 · 라벨 257,740 · 매칭된 쌍 257,740


## 3. JSON 1회 읽기 → P00 여부 + 차급 동시 판정

In [4]:
def load_json(path: Path):
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return json.loads(path.read_text(encoding=enc))
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {path}")

def find_key(obj, key):
    """중첩 dict/list에서 key를 처음 만나는 값 반환 (없으면 None)."""
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = find_key(v, key)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for item in obj:
            r = find_key(item, key)
            if r is not None:
                return r
    return None

def has_target_class(obj) -> bool:
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "classId" and isinstance(v, str) and v.strip() == TARGET_CLASS:
                return True
            if has_target_class(v):
                return True
    elif isinstance(obj, list):
        return any(has_target_class(i) for i in obj)
    return False

def normalize_class(v):
    v = v.strip() if isinstance(v, str) else ""
    return v if v in VALID_CLASSES else "미분류"

# (img, lbl, 차급) 유지 목록
keep = []
raw_class_counter = Counter()   # 원본 LargeCategoryId 값 분포 (미분류 원인 파악용)
n_drop = n_err = 0

for img, lbl in tqdm(paired, desc="판정"):
    try:
        data = load_json(lbl)
    except Exception:
        n_err += 1
        continue
    if REQUIRE_P00 and not has_target_class(data):
        n_drop += 1
        continue
    raw = find_key(data, "LargeCategoryId")
    raw_class_counter[raw if raw else "(없음)"] += 1
    keep.append((img, lbl, normalize_class(raw)))

print(f"유지 {len(keep):,} · 제외(P00없음) {n_drop:,} · 읽기오류 {n_err:,}")

판정: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 257740/257740 [01:17<00:00, 3310.62it/s]

유지 89,506 · 제외(P00없음) 168,234 · 읽기오류 0


## 4. 차급별 개수 요약

In [5]:
by_class = Counter(cls for _, _, cls in keep)

print("=" * 34)
print(f"{'차급':<10}{'쌍 수':>12}")
print("-" * 34)
for cls in VALID_CLASSES + ["미분류"]:
    print(f"{cls:<10}{by_class.get(cls, 0):>12,}")
print("-" * 34)
print(f"{'합계':<10}{len(keep):>12,}")
print("=" * 34)

# '미분류'가 있으면 원본 값 분포를 확인 (오타/신규 차급 등)
if by_class.get("미분류"):
    print("\n[미분류] 원본 LargeCategoryId 값 분포:")
    for val, n in raw_class_counter.items():
        if (val if val != "(없음)" else "") not in VALID_CLASSES:
            print(f"  {val!r}: {n:,}")

차급                 쌍 수
----------------------------------
경차               7,742
소형차             12,952
중형차             39,491
대형차             29,321
미분류                  0
----------------------------------
합계              89,506


## 5. 옮길 작업 목록 만들기

목적지: `DEST_ROOT/<차급>/원천데이터/<원본 상대경로>` (라벨은 `라벨링데이터`)

In [6]:
def dest_of(src: Path, cls: str) -> Path:
    for base, sub in ((IMAGE_ROOT, "원천데이터"), (LABEL_ROOT, "라벨링데이터")):
        try:
            return DEST_ROOT / cls / sub / src.relative_to(base)
        except ValueError:
            continue
    return DEST_ROOT / cls / "기타" / src.name

jobs = []   # (원본, 목적지)
for img, lbl, cls in keep:
    jobs.append((img, dest_of(img, cls)))
    jobs.append((lbl, dest_of(lbl, cls)))

print(f"총 옮길 파일: {len(jobs):,}개 (= 유지 {len(keep):,}쌍 × 2)")
for src, dst in jobs[:3]:
    print("  ", src, "\n    →", dst)

총 옮길 파일: 179,012개 (= 유지 89,506쌍 × 2)
   C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_T_02_001.jpg 
    → C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터\중형차\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_T_02_001.jpg
   C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\라벨링데이터\TL1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_T_02_001.json 
    → C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터\중형차\라벨링데이터\TL1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_T_02_001.json
   C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_T_02_003.jpg 
    → C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터\중형차\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_T_02_003.jpg


## 6. 실제 실행 (복사 / 이동)

`DRY_RUN = True`면 파일을 건드리지 않습니다. 실행하려면 1번 셀에서 `DRY_RUN = False`로 바꾸세요.
이미 목적지에 있는 파일은 건너뛰므로 재실행하면 이어서 진행됩니다.

In [7]:
print(f"현재 설정 → ACTION={ACTION} · DRY_RUN={DRY_RUN} · REQUIRE_P00={REQUIRE_P00}")

if DRY_RUN:
    print("\n" + "!" * 46)
    print("!! 미리보기 모드 — 실제로는 아무 파일도 옮기지 않았습니다.")
    print("!! 실행하려면 1번 셀에서 DRY_RUN = False 로 바꾸세요.")
    print("!" * 46)
else:
    ok = skip = fail = 0
    errors = []
    for src, dst in tqdm(jobs, desc=f"{ACTION} 진행"):
        try:
            if not src.exists() or dst.exists():
                skip += 1
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            (shutil.copy2 if ACTION == "copy" else shutil.move)(str(src), str(dst))
            ok += 1
        except Exception as e:
            fail += 1
            errors.append((src, str(e)))
    print(f"\n완료 · 처리 {ok:,} / 건너뜀 {skip:,} / 실패 {fail:,}")
    print("저장 위치:", DEST_ROOT)
    if errors:
        print("실패 예시:", errors[0][0], "→", errors[0][1])

현재 설정 → ACTION=copy · DRY_RUN=False · REQUIRE_P00=True


copy 진행: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 179012/179012 [13:11<00:00, 226.30it/s]


완료 · 처리 179,012 / 건너뜀 0 / 실패 0
저장 위치: C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터


## 7. 결과 확인 (차급별)

In [8]:
print(f"{'차급':<10}{'이미지':>10}{'라벨':>10}")
print("-" * 30)
for cls in VALID_CLASSES + ["미분류"]:
    base = DEST_ROOT / cls
    n_img = len(find_all_files(base / "원천데이터", IMAGE_EXTS)) if (base / "원천데이터").exists() else 0
    n_lbl = len(find_all_files(base / "라벨링데이터", {".json"})) if (base / "라벨링데이터").exists() else 0
    if n_img or n_lbl:
        print(f"{cls:<10}{n_img:>10,}{n_lbl:>10,}")

차급               이미지        라벨
------------------------------
경차             7,742     7,742
소형차           12,952    12,952
중형차           39,491    39,491
대형차           29,321    29,321
